In [1]:
# ============================================================
# IHSG FEAR & GREED INDEX - FASE 1 + 2
# ============================================================

# -- 1. Import library --
import yfinance as yf
import pandas as pd
import numpy as np
from pytrends.request import TrendReq

# -- 2. Download semua data --
print("Downloading data...")
ihsg        = yf.download("^JKSE", period="5y", progress=False)
idr         = yf.download("IDR=X", period="5y", progress=False)["Close"].squeeze()

pytrends    = TrendReq(hl='id-ID', tz=420)
pytrends.build_payload(["IHSG","rupiah","saham","investasi","bursa"], timeframe='today 5-y', geo='ID')
trends_data = pytrends.interest_over_time()
if 'isPartial' in trends_data.columns:
    trends_data = trends_data.drop(columns=['isPartial'])

print("✅ Data berhasil didownload!")

# -- 3. Definisi fungsi --
def hitung_rsi(data, period=14):
    delta     = data.diff()
    naik      = delta.clip(lower=0)
    turun     = -delta.clip(upper=0)
    avg_naik  = naik.ewm(com=period-1, adjust=False).mean()
    avg_turun = turun.ewm(com=period-1, adjust=False).mean()
    rs        = avg_naik / avg_turun
    return 100 - (100 / (1 + rs))

def normalisasi(series, window=90):
    recent  = series.dropna().tail(window)
    min_val = float(recent.min())
    max_val = float(recent.max())
    nilai   = float(series.dropna().iloc[-1])
    if max_val == min_val:
        return 50.0
    return round((nilai - min_val) / (max_val - min_val) * 100, 1)

def get_label(skor):
    if skor >= 75: return "🔴 Extreme Greed"
    if skor >= 55: return "🟠 Greed"
    if skor >= 45: return "🟡 Neutral"
    if skor >= 25: return "🟢 Fear"
    return "🔵 Extreme Fear"

# -- 4. Hitung semua indikator --
close     = ihsg["Close"].squeeze()
volume    = ihsg["Volume"].squeeze()

# Momentum
ma50      = close.rolling(window=50).mean()
momentum  = (close - ma50) / ma50 * 100

# RSI
rsi       = hitung_rsi(close)

# Volume ratio
vol_5     = volume.rolling(5).mean()
vol_90    = volume.rolling(90).mean()
vol_ratio = vol_5 / vol_90

# Rupiah (inverted)
idr_change   = idr.pct_change(10)
idr_inverted = -idr_change

# Google Trends
bobot_keyword = {"IHSG":0.30,"rupiah":0.30,"saham":0.20,"investasi":0.15,"bursa":0.05}
trends_skor   = sum(trends_data[kw] * bobot for kw, bobot in bobot_keyword.items())
trends_daily  = trends_skor.resample("D").ffill()
trends_aligned = trends_daily.reindex(close.index, method="ffill")

# -- 5. Normalisasi & skor final --
skor_momentum = normalisasi(momentum)
skor_rsi      = normalisasi(rsi)
skor_volume   = normalisasi(vol_ratio)
skor_rupiah   = normalisasi(idr_inverted)
skor_trends   = normalisasi(trends_aligned)

skor_final = round(
    skor_momentum * 0.25 +
    skor_rsi      * 0.20 +
    skor_volume   * 0.20 +
    skor_rupiah   * 0.20 +
    skor_trends   * 0.15,
    1
)

# -- 6. Output --
print("\n=== Skor per indikator (0-100) ===")
print(f"Momentum  : {skor_momentum}")
print(f"RSI       : {skor_rsi}")
print(f"Volume    : {skor_volume}")
print(f"Rupiah    : {skor_rupiah}")
print(f"Trends    : {skor_trends}")
print(f"\n{'='*35}")
print(f"  IHSG Fear & Greed Index")
print(f"  Skor   : {skor_final} / 100")
print(f"  Status : {get_label(skor_final)}")
print(f"{'='*35}")

✅ Data berhasil didownload!

=== Skor per indikator (0-100) ===
Momentum  : 35.1
RSI       : 17.6
Volume    : 35.4
Rupiah    : 12.5
Trends    : 37.6

  IHSG Fear & Greed Index
  Skor   : 27.5 / 100
  Status : 🟢 Fear


In [2]:
import os
from datetime import datetime

# Hitung skor historis per hari (bukan cuma hari ini)
def hitung_skor_harian(date_index):
    hasil = []
    
    for i, date in enumerate(date_index):
        # Minimal butuh 100 data sebelumnya buat indikator
        if i < 100:
            continue
        
        # Slice data sampai tanggal ini
        c   = close.iloc[:i+1]
        v   = volume.iloc[:i+1]
        idr = idr_change.iloc[:i+1]
        tr  = trends_aligned.iloc[:i+1]
        
        # Hitung indikator
        ma   = c.rolling(50).mean()
        mom  = (c - ma) / ma * 100
        rs   = hitung_rsi(c)
        vr   = v.rolling(5).mean() / v.rolling(90).mean()
        idri = -idr
        
        # Normalisasi
        s_mom = normalisasi(mom)
        s_rsi = normalisasi(rs)
        s_vol = normalisasi(vr)
        s_idr = normalisasi(idri)
        s_tr  = normalisasi(tr)
        
        skor = round(
            s_mom * 0.25 +
            s_rsi * 0.20 +
            s_vol * 0.20 +
            s_idr * 0.20 +
            s_tr  * 0.15,
            1
        )
        
        hasil.append({
            "date"  : date,
            "skor"  : skor,
            "close" : float(c.iloc[-1])
        })
    
    return pd.DataFrame(hasil).set_index("date")

print("Menghitung skor historis... (ini agak lama ~30 detik)")
df_historis = hitung_skor_harian(close.index)

# Simpan ke CSV
os.makedirs("../data", exist_ok=True)
df_historis.to_csv("../data/historis.csv")

print(f"✅ Selesai! {len(df_historis)} hari data tersimpan")
print(df_historis.tail())

Menghitung skor historis... (ini agak lama ~30 detik)
✅ Selesai! 1099 hari data tersimpan
            skor        close
date                         
2026-04-24  43.5  7129.490234
2026-04-27  43.7  7106.520020
2026-04-28  41.6  7072.392090
2026-04-29  39.9  7101.226074
2026-04-30  35.9  6956.804199
